# 05 — Spécialiser la Base avec le LoRA Monster

**Objectif :** relier le dataset Monster à une révision précise de la Base, puis comprendre le cache et l'entraînement du diffuseur.

~~~text
Base retenue et projecteur figé ──────┐
                                    ├─► features / latents Monster ─► LoRA ─► comparaison avant / après
Dataset Monster versionné et audité ─┘
~~~

Le nouvel entraînement Monster est une étape distincte. La présence du dataset prêt ne signifie pas que de nouveaux poids existent. L'ancien LoRA publié référence l'ancien projecteur ; il ne devient pas automatiquement compatible avec la nouvelle Base.

**Préparation uniquement : aucun nouvel entraînement LoRA ni pré-calcul GPU Monster avant une nouvelle demande explicite de l’utilisateur.** Les cellules ci-dessous lisent les données et affichent des recettes ; elles ne lancent pas ces calculs.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, preview_script,
)
print("Projet :", PROJECT.name)


In [ ]:
dataset, pointer = current_dataset()
base_release = ARTIFACTS / "spatial/release"
projector = base_release / "projector"
base_validation = ARTIFACTS / "monster/base-validation.json"
projector_sha = sha256(projector / "model.safetensors") if (projector / "model.safetensors").exists() else None
cache = PROJECT / "cache" / ("monster-" + dataset.name + "-" + (projector_sha[:12] if projector_sha else "pending"))
training_cache = ARTIFACTS / "monster/sprite-training-256.json"
monster_output = REPO / "1-local-generation/weights" / ("sharm-monster-" + dataset.name)
gate = read_json(base_validation, {})
print("Dataset :", dataset.name)
print("Projecteur :", projector_sha or "non collecté")
print("Validation correspondant au projecteur :", bool(projector_sha and gate.get("passed") and gate.get("projector_sha256") == projector_sha))
print("Nouveau LoRA terminé :", (monster_output / "training-complete.json").exists())

## Cache lié aux données et au projecteur

Les légendes uniques sont encodées par le 4B FP8, puis projetées. Les images RGBA produisent leurs latents VAE. L'identité du cache contient les empreintes des images, du corpus et du projecteur ; un changement de Base impose de recalculer les features projetées.

Les scripts exigent un rapport de validation de Base avec `passed` et le SHA du projecteur. **Ne pas fabriquer ce rapport à partir de la seule fin de l'entraînement ou du seul contrôle du loader.** La nouvelle Base doit réellement être retenue et sa compatibilité d'entraînement confirmée avant de préparer les caches.

Ici, le calcul se fait en 256 px sur des copies agrandies des sprites ; la cible de présentation reste 128 × 128. Une expérience 512 px demande son propre cache et ses mesures.

In [ ]:
# Aperçu de la recette future, sans calcul GPU ni création de cache.
preview_script("01-encoder-distillation/cache_features.py", [
    "student", "--corpus", dataset / "caption-corpus.jsonl", "--cache-root", cache, "--batch-size", 1,
])
preview_script("02-sprite-lora/prepare_training_cache.py", [
    "--resolution", 256, "--dataset-root", dataset, "--cache-root", cache,
    "--projector-dir", projector, "--base-validation", base_validation,
    "--output-report", training_cache,
])

## Paramètres de départ du LoRA

Le rang contrôle la taille de l'adaptation. Le batch 1 et l'accumulation 4 donnent quatre exemples par mise à jour. Le checkpointing alterné économise la mémoire. Les 500 pas ci-dessous reprennent la recette historique comme point de départ ; leur adéquation au nouveau dataset devra être évaluée, pas supposée.

Le diffuseur est chargé en NF4 pour l'apprentissage, le calcul se fait en BF16 et l'optimiseur est en 8 bits. Le VL demeure gelé en FP8. Les nouvelles sorties utilisent un dossier distinct pour préserver le LoRA historique.

In [ ]:
# Paramètres de la recette future ; cette cellule affiche seulement la commande.
preview_script("02-sprite-lora/train_lora.py", [
    "--resolution", 256, "--training-cache", training_cache,
    "--base-validation", base_validation, "--steps", 500, "--accumulation", 4,
    "--rank", 8, "--learning-rate", 0.0001, "--checkpointing-mode", "alternating",
    "--output", monster_output,
])

In [ ]:
completion = read_json(monster_output / "training-complete.json")
if completion:
    print(__import__("json").dumps(completion, ensure_ascii=False, indent=2))
    history = read_jsonl(monster_output / "history.jsonl")
    table([{
        "pas": r.get("step"),
        "validation flow MSE": r.get("validation_flow_mse"),
    } for r in history], ["pas", "validation flow MSE"])
else:
    print("Aucun résultat du nouveau LoRA disponible dans ce dossier.")

## Comparaison et publication du nouveau Monster

Réserver les espèces de validation ; comparer Base seule et Base + LoRA avec les mêmes prompts, graines, tailles et paramètres. Montrer les PNG RGBA 128 px et leurs aperçus sur damier. Vérifier silhouette, identité, style et alpha ; ne pas retenir seulement les meilleurs exemples.

Le script historique `evaluate_sprites.py` et le publisher historique utilisent leurs anciens chemins. Ils devront être adaptés au nouveau dataset et à la Base retenue avant de servir à un nouveau Monster ; ce notebook ne les lance pas sur d'anciens caches en les présentant comme de nouveaux résultats.

Le futur export Monster doit référencer la révision HF immuable de la Base, le SHA du projecteur, celui du LoRA et celui du dataset. La publication de Base autorisée actuellement ne publie pas à elle seule un nouveau Monster.

**Suite :** [06 — Export, publication et versionnement](06_publication_et_versions.ipynb).